In [331]:
import matplotlib.pyplot as plt
import ast
import pandas as pd
import numpy as np
import cnn_surgery.utils.metrics as metrics

In [332]:
DATASET = 'mnist'
file_path = f'{DATASET}_relative_acc_evaluation.csv'
df = pd.read_csv(file_path)

In [333]:
# filters
df_filtered = df
df_filtered = df[~df['original_accuracy'].apply(lambda x: 0.0 in ast.literal_eval(x))]
#df_filtered = df_filtered[~df_filtered.apply(lambda row: ast.literal_eval(row['original_accuracy'])[row['target_class']] < 0.9, axis=1)]
#df_filtered = df_filtered[~df_filtered.apply(lambda row: np.mean(ast.literal_eval(row['original_accuracy'])) < 0.7, axis=1)]
df_filtered = df_filtered[~(df_filtered['distance_travelled'] < 1.0)]
df = df_filtered

In [334]:
df['target_accuracy_before'] = df.apply(lambda row: ast.literal_eval(row['original_accuracy'])[row['target_class']], axis=1)
df['target_accuracy_after'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after'])[row['target_class']], axis=1)
df['target_accuracy_rv'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after_rv'])[row['target_class']], axis=1)
df['target_accuracy_fa'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after_fa'])[row['target_class']], axis=1)

df['relative_target_accuracy_after'] = df['target_accuracy_after'] / df['target_accuracy_before']
df['relative_target_accuracy_rv'] = df['target_accuracy_rv'] / df['target_accuracy_before']
df['relative_target_accuracy_fa'] = df['target_accuracy_fa'] / df['target_accuracy_before']

df['mean_retain_accuracy_before'] = df.apply(lambda row: np.mean([ast.literal_eval(row['original_accuracy'])[i] for i in range(len(ast.literal_eval(row['original_accuracy']))) if i != row['target_class']]), axis=1)
df['mean_retain_accuracy_after'] = df.apply(lambda row: np.mean([ast.literal_eval(row['accuracy_after'])[i] for i in range(len(ast.literal_eval(row['original_accuracy']))) if i != row['target_class']]), axis=1)
df['mean_retain_accuracy_rv'] = df.apply(lambda row: np.mean([ast.literal_eval(row['accuracy_after_rv'])[i] for i in range(len(ast.literal_eval(row['original_accuracy']))) if i != row['target_class']]), axis=1)
df['mean_retain_accuracy_fa'] = df.apply(lambda row: np.mean([ast.literal_eval(row['accuracy_after_fa'])[i] for i in range(len(ast.literal_eval(row['original_accuracy']))) if i != row['target_class']]), axis=1)

df['relative_mean_retain_accuracy_after'] = df['mean_retain_accuracy_after'] / df['mean_retain_accuracy_before']
df['relative_mean_retain_accuracy_rv'] = df['mean_retain_accuracy_rv'] / df['mean_retain_accuracy_before']
df['relative_mean_retain_accuracy_fa'] = df['mean_retain_accuracy_fa'] / df['mean_retain_accuracy_before']

In [335]:
groups = df.groupby('stop_threshold')
tables = {}
relative = True

if relative:
    relative = 'relative_'
else:
    relative = ''

for group in groups:
    target_class, group_df = group  # Unpack the tuple
    table = pd.DataFrame({
        'before': {
            'target': group_df['target_accuracy_before'].mean(),
            'control': group_df['mean_retain_accuracy_before'].mean(),
        },
        'Gradient Ascent': {
            'target': group_df[relative + 'target_accuracy_fa'].mean(),
            'control': group_df[relative + 'mean_retain_accuracy_fa'].mean(),
        },
        'Random Vector': {
            'target': group_df[relative + 'target_accuracy_rv'].mean(),
            'control': group_df[relative + 'mean_retain_accuracy_rv'].mean(),
        },
        'Our Method': {
            'target': group_df[relative + 'target_accuracy_after'].mean(),
            'control': group_df[relative + 'mean_retain_accuracy_after'].mean(),
        },
    })
    tables[target_class] = table

In [336]:
for table in tables:
    print(f'Target Class: {table}')
    print(tables[table].T.round(2))

Target Class: 0.1
                 target  control
before             0.76     0.80
Gradient Ascent    0.03     0.36
Random Vector      1.04     0.82
Our Method         0.44     0.65
Target Class: 0.2
                 target  control
before             0.77     0.82
Gradient Ascent    0.03     0.34
Random Vector      1.10     0.88
Our Method         0.53     0.72
Target Class: 0.3
                 target  control
before             0.80     0.83
Gradient Ascent    0.03     0.31
Random Vector      1.14     0.89
Our Method         0.59     0.75
Target Class: 0.4
                 target  control
before             0.83     0.84
Gradient Ascent    0.02     0.28
Random Vector      1.14     0.90
Our Method         0.62     0.76
Target Class: 0.5
                 target  control
before             0.84     0.85
Gradient Ascent    0.02     0.28
Random Vector      1.16     0.91
Our Method         0.64     0.78
Target Class: 0.6
                 target  control
before             0.85     0.85
G

In [337]:
print(df.target_accuracy_before.mean())
print(df.mean_retain_accuracy_before.mean())

0.821638402964117
0.8411779195250446


In [338]:
df['target_over_overall'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after'])[row['target_class']], axis=1) / df['overall_accuracy']
df['target_over_overall_rv'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after_rv'])[row['target_class']], axis=1) / df['overall_accuracy_rv']
df['mean_rest_acc'] = df.apply(lambda row: np.mean([x for i,x in enumerate(ast.literal_eval(row['accuracy_after'])) if i != row['target_class']]), axis=1)
df['mean_rest_acc_rv'] = df.apply(lambda row: np.mean([x for i,x in enumerate(ast.literal_eval(row['accuracy_after_rv'])) if i != row['target_class']]), axis=1)
df['avg_retain_accuracy'] = df.apply(lambda row: metrics.avg_retain_accuracy(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class'], proportional=True), axis=1)
df['avg_retain_accuracy_rv'] = df.apply(lambda row: metrics.avg_retain_accuracy(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after_rv']), row['target_class'], proportional=True), axis=1)
# calculate metrics
df['max_difference'] = df.apply(lambda row: metrics.max_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
df['mean_difference'] = df.apply(lambda row: metrics.mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
df['clipped_negative_mean_difference'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
#df['relative_clipped_negative_mean_difference'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class'], proportional=True), axis=1)
df['target_difference'] = df.apply(lambda row: metrics.target_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
df['relative_target_difference'] = df.apply(lambda row: metrics.target_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class'], proportional=True), axis=1)
df['relative_target_difference_rv'] = df.apply(lambda row: metrics.target_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after_rv']), row['target_class'], proportional=True), axis=1)
df['target_accuracy'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after'])[row['target_class']], axis=1)
# metrics for rv
df['max_difference_rv'] = df.apply(lambda row: metrics.max_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after_rv']), row['target_class']), axis=1)
df['mean_difference_rv'] = df.apply(lambda row: metrics.mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after_rv']), row['target_class']), axis=1)
df['clipped_negative_mean_difference_rv'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after_rv']), row['target_class']), axis=1)
df['target_difference_rv'] = df.apply(lambda row: metrics.target_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after_rv']), row['target_class']), axis=1)
df['target_accuracy_rv'] = df.apply(lambda row: ast.literal_eval(row['accuracy_after_rv'])[row['target_class']], axis=1)

In [339]:
df.target_accuracy.mean()
df.mean_rest_acc.mean()

0.6504391754253676

In [340]:
for col in df.columns:
    if 'fa' in col: print(col)

accuracy_after_fa
overall_accuracy_fa
target_accuracy_fa
relative_target_accuracy_fa
mean_retain_accuracy_fa
relative_mean_retain_accuracy_fa


In [341]:
df

,model_idx,original_accuracy,accuracy_after,overall_accuracy,target_class,dataset,lr,stop_threshold,l2_penalty,loss_fn,...,mean_difference,clipped_negative_mean_difference,target_difference,relative_target_difference,relative_target_difference_rv,target_accuracy,max_difference_rv,mean_difference_rv,clipped_negative_mean_difference_rv,target_difference_rv
2,2,"[0.9826531, 0.9920705, 0.9573643, 0.96039605, ...","[0.6540816326530612, 0.9920704845814978, 0.911...",0.8761,4,mnist,0.1,0.1,0.0,boost,...,-0.076103,-0.077611,0.023422,0.024008,-6.263060e-03,0.952138,-0.049231,-0.019537,-0.014185,-6.109991e-03
4,4,"[0.9459184, 0.9929515, 0.96802324, 0.929703, 0...","[0.5836734693877551, 0.0, 0.0, 0.0, 0.01425661...",0.2395,4,mnist,0.1,0.1,0.0,boost,...,0.265309,0.265309,0.949083,0.985201,5.105708e-01,0.014257,0.072051,0.337602,0.335111,4.918533e-01
5,5,"[0.99081635, 0.99030834, 0.93798447, 0.9386138...","[0.9071428571428571, 0.9541850220264317, 0.906...",0.8986,4,mnist,0.1,0.1,0.0,boost,...,-0.045447,-0.050759,0.020367,0.021164,6.031747e-02,0.941955,0.024971,0.048493,0.046625,5.804481e-02
6,6,"[0.9612245, 0.9929515, 0.91957366, 0.95445544,...","[0.9061224489795918, 0.9215859030837005, 0.937...",0.9178,4,mnist,0.1,0.1,0.0,boost,...,-0.008924,-0.017064,0.010183,0.010977,1.295280e-01,0.917515,0.075265,0.115536,0.108649,1.201629e-01
7,7,"[0.9867347, 0.9920705, 0.9282946, 0.9049505, 0...","[0.9653061224489796, 0.9894273127753304, 0.913...",0.8899,4,mnist,0.1,0.1,0.0,boost,...,-0.030827,-0.029043,-0.005092,-0.005230,1.046044e-03,0.978615,-0.025144,-0.003017,-0.006294,1.018348e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790,190,"[0.96836734, 0.98678416, 0.93507755, 0.9326733...","[0.8806122448979592, 0.0, 0.502906976744186, 0...",0.5910,4,mnist,0.1,0.9,0.0,boost,...,0.225064,0.219930,0.551935,0.572334,-6.335799e-03,0.412424,-0.022537,-0.007730,-0.004493,-6.109982e-03
1792,192,"[0.97755104, 0.99030834, 0.9563953, 0.9663366,...","[0.43673469387755104, 0.0, 0.0, 0.0, 0.0295315...",0.1732,4,mnist,0.1,0.9,0.0,boost,...,0.151890,0.147327,0.922607,0.968984,8.556157e-03,0.029532,-0.099881,-0.030506,-0.032156,8.146647e-03
1793,193,"[0.97959185, 0.98678416, 0.93604654, 0.9306931...","[0.9836734693877551, 0.000881057268722467, 0.4...",0.5833,4,mnist,0.1,0.9,0.0,boost,...,-0.193190,-0.195625,0.181263,0.189765,-2.345418e-02,0.773931,-0.040225,-0.027349,-0.006805,-2.240328e-02
1795,195,"[0.9897959, 0.9938326, 0.97771317, 0.9722772, ...","[0.9826530612244898, 0.9894273127753304, 0.958...",0.9649,4,mnist,0.1,0.9,0.0,boost,...,-0.016405,-0.011693,-0.007128,-0.007353,-1.890756e-09,0.976578,-0.012245,-0.000474,-0.002314,-1.832994e-09
